# Combined Claims Features (Rx + Lab + Patient Stage) — All Cohorts

Cohort-based refactor of `04_claims_monthly` (NBA 3.0, Wenjie Chen). Sections 1–3 walk through every table the original uses; sections 4–6 rebuild the same features per cohort (windows of 1, 3, 6, 12 months ending at each cutoff, `RUNNING_`, `PCT_`), write one row per NPI per `COHORT_ID` to `claims_monthly_cohort` and `final_claims_monthly_cohort`, and validate against the original.

Source: `/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/03_Feature_Engineering/03_HCP_Level_Features/04_claims_monthly`

## What this notebook does, in plain words

Notebooks 01–03 each count **one** claim type. This one **mixes** them: it starts from every **Rx (prescription) claim** of an HCP and asks *"what did we know about that patient's kidneys when the prescription was written?"*

For every Rx claim it attaches three kinds of patient information and then counts claims per HCP:

| Feature family | Question it answers | Extra table joined to the Rx claim | Column prefix |
| --- | --- | --- | --- |
| A. Patient CKD stage | What GFR / UACR stage was the patient in that month? | `pat_gfr_dx_stage_and_treatment_category` (patient x month) | `TOTAL_RX_CLAIMS_PATIENT_WITH_GFR_...` |
| B. Same-day lab test | Did the patient have a GFR / UACR lab test **on the same day**, and what stage did it show, split by the drug class? | `Lab_04` (lab results) | `TOTAL_RX_CLAIMS_PATIENT_WITH_TEST_...` |
| C. Stage x treatment | Which treatment categories / brands is the patient on in that month, split by stage? | `pat_uacr_stage_and_treatment_category_new` (patient x month x treatment) | `TOTAL_RX_CLAIMS_PATIENT_WITH_DX_...` |

Plus `TOTAL_RX_CLAIMS` (all Rx claims, the denominator for the `PCT_` columns). Then the same steps as the other notebooks: fill missing months, sum over the last 1/3/6/12 months, `RUNNING_` totals, `PCT_` shares, map NPI to `PROVIDER_ID` and write `final_claims_monthly`.

Because every stage is crossed with every treatment category / brand, the output is very wide (about 3,700 columns).

**How to read this notebook:** section 1 shows each input table as it is stored; section 2 shows the claim-level tables the original builds by joining them (the step just before counting); section 3 shows the original output. Everything is shown for the 10 test HCPs (validation mode).

## 0. Connection and Config

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "../00_cohort_config"

## 1. The input tables

The original reads five tables. The cell below loads them; each following cell shows one of them.

* **Rx claims** are filtered to the 10 test HCPs.
* **Lab tests and patient tables** are keyed on the *patient*, not the HCP, so they are shown only for the patients of those HCPs' Rx claims.
* In validation mode the two patient tables come from the 3.1 patient journey (`kerendia_nba_3_1_pat_gfr_dx_stage_validation`, `kerendia_nba_3_1_pat_uacr_stage_validation`); they have the same columns as the 3.0 tables the original reads.

In [0]:
import builtins
# Paths, brands, windows and patient tables come from 00_cohort_config (HCP shared constants)
_QUERY_OUTPUT    = HCP_QUERY_OUTPUT
_NOTEBOOK_OUTPUT = HCP_NOTEBOOK_OUTPUT

# Rx claims: the base of every feature. Same cleaning as the original (trailing spaces removed
# from TREATMENT_CATEGORY). Validation mode keeps only the 10 test HCPs.
df_rx_raw = spark.read.format("delta").load(_QUERY_OUTPUT + "Rx_01_new")
df_rx = (apply_npi_filter(df_rx_raw, NPI_COL)
         .withColumn("TREATMENT_CATEGORY", F.regexp_replace("TREATMENT_CATEGORY", r"[\s]*$", "")))
# df_rx keeps the full history (walkthrough and offset-0 parity use it, as the original).
# DATA_START_DATE (team rule, RUNNING_ included) is applied to the feature rows (cell "Feature rows
# (once) and cohort list") and in the HCP trace.

# Lab tests: joined to Rx by patient + same service date. As in the original, PATIENT_ID is cast
# to string and the lab's NPI is dropped (the lab can come from any HCP).
df_lab_raw = spark.read.format("delta").load(_QUERY_OUTPUT + "Lab_04")
df_lab = df_lab_raw.withColumn("PATIENT_ID", F.col("PATIENT_ID").cast("string")).drop("NPI_NUMBER")

# Dx claims: read by the original but not used by any feature
df_dx_raw = spark.read.format("delta").load(_QUERY_OUTPUT + "Dx_03")

# Patient tables (one row per patient per month; the UACR one per patient x month x treatment)
_GFR_TABLE  = get_patient_table("gfr_dx_stage")
_UACR_TABLE = get_patient_table("uacr_stage")
pat_gfr  = spark.read.format("delta").load(_QUERY_OUTPUT + _GFR_TABLE)
pat_uacr = spark.read.format("delta").load(_QUERY_OUTPUT + _UACR_TABLE)

# Rows of the patient-keyed tables that belong to the patients of this run's Rx claims
_rx_patients  = df_rx.select("PATIENT_ID").distinct()
df_lab_view   = df_lab.join(_rx_patients, on="PATIENT_ID", how="left_semi")
pat_gfr_view  = pat_gfr.join(_rx_patients, on="PATIENT_ID", how="left_semi")
pat_uacr_view = pat_uacr.join(_rx_patients, on="PATIENT_ID", how="left_semi")

print(f"Patient GFR table : {_GFR_TABLE}")
print(f"Patient UACR table: {_UACR_TABLE}")
print(f"Rx claims: {df_rx.count():,} rows | patients: {_rx_patients.count():,}")
print(f"Lab tests of those patients: {df_lab_view.count():,} | GFR rows: {pat_gfr_view.count():,} | "
      f"UACR rows: {pat_uacr_view.count():,}")

### 1.1 Rx claims (`Rx_01_new`)
One row per prescription: which HCP (`NPI_NUMBER`) wrote which drug for which patient on which day.
* `TREATMENT_CATEGORY` = drug class (e.g. SGLT2, ACE/ARB, MRA). Used by family B.
* `CATEGORY` = brand-level group (e.g. FARXIGA, JARDIANCE). Family B uses it for the brand columns.
* Every Rx claim counts once in `TOTAL_RX_CLAIMS`; families A–C split these claims by patient information.

Tables below: sample claims, claims per drug class, claims per drug class x brand group.

In [0]:
print(f"Rx_01_new: {len(df_rx_raw.columns)} columns | rows for the HCPs in this run: {df_rx.count():,}")
display(df_rx.orderBy(NPI_COL, SVC_COL).limit(50))
display(df_rx.groupBy("TREATMENT_CATEGORY").count().orderBy(F.desc("count")))
display(df_rx.groupBy("TREATMENT_CATEGORY", "CATEGORY").count().orderBy("TREATMENT_CATEGORY", F.desc("count")))

### 1.2 Lab tests (`Lab_04`)
One row per lab result: patient, test date, `TEST_TYPE` (GFR or UACR), the result and the `STAGE` it implies.
* GFR stages are text like `CHRONIC KIDNEY DISEASE, STAGE 3 (MODERATE)`; UACR stages are `A1`/`A2`/`A3`.
* Family B looks for a lab test of the same patient **on the same day** as the Rx claim.

Tables below: sample lab rows of this run's patients, and rows per test type x stage.

In [0]:
print(f"Lab_04: {len(df_lab_raw.columns)} columns | rows for this run's patients: {df_lab_view.count():,}")
display(df_lab_view.orderBy("PATIENT_ID", SVC_COL).limit(50))
display(df_lab_view.groupBy("TEST_TYPE", "STAGE").count().orderBy("TEST_TYPE", "STAGE"))

### 1.3 Patient CKD stage per month (`pat_gfr_dx_stage_and_treatment_category`)
One row per patient per month, built by the patient journey notebooks.
* `GFR_DX_STAGE` (1–5) and `UACR_STAGE` (1–3) = the patient's kidney stage that month. Family A uses these two.
* `GFR_UACR_DX_STAGE`, `CATEGORY` and `BRAND` (lists) are in the table but not used by this notebook.

Tables below: sample rows, rows per GFR x UACR stage, and a uniqueness check (one row per patient-month means the join cannot duplicate Rx claims).

In [0]:
display(pat_gfr_view.orderBy("PATIENT_ID", "SVC_DT_M").limit(50))
display(pat_gfr_view.groupBy("GFR_DX_STAGE", "UACR_STAGE").count().orderBy("GFR_DX_STAGE", "UACR_STAGE"))
_n, _k = pat_gfr_view.count(), pat_gfr_view.select("PATIENT_ID", "SVC_DT_M").distinct().count()
print(f"Rows: {_n:,} | distinct (patient, month): {_k:,} | duplicates: {_n - _k:,}")

### 1.4 Patient stage x treatment per month (`pat_uacr_stage_and_treatment_category_new`)
Same stage columns as 1.3, but `CATEGORY` and `BRAND` are single values instead of lists. The two lists were expanded **independently**, so every category is paired with every brand: a patient on 3 treatments in a month has 3 x 3 = **9 rows** for that month (row counts per patient-month are 1, 4, 9, 16, 25, ...). The 3.0 table the original reads has the same structure.
* Family C joins this table to the Rx claims, so one Rx claim turns into one row per (category, brand) pair of the patient that month.

Tables below: sample rows, how many rows each patient-month has, and rows per treatment category and brand.

In [0]:
display(pat_uacr_view.orderBy("PATIENT_ID", "SVC_DT_M", "CATEGORY").limit(50))
display(pat_uacr_view.groupBy("PATIENT_ID", "SVC_DT_M").count()
        .groupBy(F.col("count").alias("ROWS_PER_PATIENT_MONTH")).count()
        .withColumnRenamed("count", "PATIENT_MONTHS").orderBy("ROWS_PER_PATIENT_MONTH"))
display(pat_uacr_view.groupBy("CATEGORY").count().orderBy(F.desc("count")))
display(pat_uacr_view.groupBy("BRAND").count().orderBy(F.desc("count")))

### 1.5 Dx claims (`Dx_03`)
The original loads `Dx_03` (the table notebook 03 is built from) but never uses it in a feature; it is only deleted at the end. It can be dropped in the refactor.

## 2. What the original builds from them (claim level, before counting)

Each family joins extra information onto the Rx claims and then counts claims per HCP per month for every label combination. The tables below show those joined claim-level tables, so you can see exactly what gets counted.

The claim month here is the cohort month from `00_cohort_config` (`assign_cohort_month`, offset `OFFSET_DAYS`); at offset 0 it equals the original calendar month (`date_trunc('MM', SVC_DT)`). Stage labels use the original rules: patient-table stages that are missing or out of range become `..._UNKNOWN`, lab stages that do not match become `..._UNSPECIFIED`.

In [0]:
# Same label rules as the original cells 2.1 / 2.2 / 2.3
def gfr_label(c):
    return (F.when(c == 1, "GFR_STAGE_1").when(c == 2, "GFR_STAGE_2").when(c == 3, "GFR_STAGE_3")
             .when(c == 4, "GFR_STAGE_4").when(c == 5, "GFR_STAGE_5").otherwise("GFR_STAGE_UNKNOWN"))

def uacr_label(c):
    return (F.when(c == 1, "UACR_STAGE_1").when(c == 2, "UACR_STAGE_2").when(c == 3, "UACR_STAGE_3")
             .otherwise("UACR_STAGE_UNKNOWN"))

_GFR_LAB_STAGES = {"CHRONIC KIDNEY DISEASE, STAGE 1": "GFR_STAGE_1",
                   "CHRONIC KIDNEY DISEASE, STAGE 2 (MILD)": "GFR_STAGE_2",
                   "CHRONIC KIDNEY DISEASE, STAGE 3 (MODERATE)": "GFR_STAGE_3",
                   "CHRONIC KIDNEY DISEASE, STAGE 4 (SEVERE)": "GFR_STAGE_4",
                   "CHRONIC KIDNEY DISEASE, STAGE 5": "GFR_STAGE_5"}
_UACR_LAB_STAGES = {"A1": "UACR_STAGE_1", "A2": "UACR_STAGE_2", "A3": "UACR_STAGE_3"}

def lab_gfr_label():
    e = F.lit("GFR_STAGE_UNSPECIFIED")
    for s, lab in reversed(list(_GFR_LAB_STAGES.items())):
        e = F.when((F.col("TEST_TYPE") == "GFR") & (F.col("STAGE") == s), lab).otherwise(e)
    return e

def lab_uacr_label():
    e = F.lit("UACR_STAGE_UNSPECIFIED")
    for s, lab in reversed(list(_UACR_LAB_STAGES.items())):
        e = F.when((F.col("TEST_TYPE") == "UACR") & (F.col("STAGE") == s), lab).otherwise(e)
    return e

# Rx claims with their claim month (join key to the monthly patient tables)
df_rx_m = df_rx.withColumn("SVC_DT_M", assign_cohort_month(F.col(SVC_COL)))
_n_rx = df_rx_m.count()

### 2.1 Family A: Rx claim + patient CKD stage that month
Each Rx claim gets the patient's `GFR_STAGE` and `UACR_STAGE` for the claim month (left join on patient + month). Claims of patients with no row that month get `..._UNKNOWN`.

Counted as: claims per `GFR_STAGE` (e.g. `TOTAL_RX_CLAIMS_PATIENT_WITH_GFR_STAGE_3`) and per `GFR_STAGE` x `UACR_STAGE` (e.g. `..._GFR_STAGE_3_AND_UACR_STAGE_2`).

Tables below: sample joined claims, and claims per GFR x UACR label (the counts behind these columns).

In [0]:
df_prep_a = (
    df_rx_m.select(NPI_COL, "PATIENT_ID", SVC_COL, "SVC_DT_M", "TREATMENT_CATEGORY")
    .join(pat_gfr.select("PATIENT_ID", F.to_date("SVC_DT_M").alias("SVC_DT_M"), "GFR_DX_STAGE", "UACR_STAGE"),
          on=["PATIENT_ID", "SVC_DT_M"], how="left")
    .withColumn("GFR_STAGE", gfr_label(F.col("GFR_DX_STAGE")))
    .withColumn("UACR_STAGE", uacr_label(F.col("UACR_STAGE")))
)
print(f"Rx claims: {_n_rx:,} | after join: {df_prep_a.count():,} (equal = no duplicates) | "
      f"with a patient-stage row: {df_prep_a.filter(F.col('GFR_DX_STAGE').isNotNull()).count():,}")
display(df_prep_a.orderBy(NPI_COL, SVC_COL).limit(50))
display(df_prep_a.groupBy("GFR_STAGE", "UACR_STAGE").count().orderBy("GFR_STAGE", "UACR_STAGE"))

### 2.2 Family B: Rx claim + same-day lab test
Each Rx claim is matched to lab tests of the same patient on the **same date** (left join on patient + `SVC_DT`). The lab's stage becomes `GFR_STAGE` / `UACR_STAGE` (`..._UNSPECIFIED` when there is no matching test of that type).

Counted as: claims per lab stage x drug class (e.g. `TOTAL_RX_CLAIMS_PATIENT_WITH_TEST_GFR_STAGE_3_WITH_SGLT2`) and per lab stage x brand group for 8 brands (INVOKANA 100/300MG, FARXIGA, JARDIANCE, SPIRONOLACTONE-MRA, Diuretic + MRA, DPP-IV, GLP-1).

Same-day tests are rare: `Lab_04` has about 120M GFR results but only about 15.5K UACR results, so most family B columns are 0 for most HCPs.

Points to note (to confirm with Wenjie in the refactor):
* In the original the month of this join comes from the **lab** row, so Rx claims with no same-day lab have no month and drop out of family B.
* A claim matched to two lab tests that day (e.g. a GFR and a UACR) is counted twice.
* `recency_window = 90` is defined in the original but never used: only same-day tests count.

Tables below: claim counts before/after the join, sample matched claims, and claims per lab stage x drug class.

In [0]:
df_prep_b = (
    df_rx_m.select(NPI_COL, "PATIENT_ID", SVC_COL, "SVC_DT_M", "TREATMENT_CATEGORY", "CATEGORY")
    .join(df_lab.select("PATIENT_ID", SVC_COL, "TEST_TYPE", "MEDIAN_TEST_RESULT", "STAGE"),
          on=[SVC_COL, "PATIENT_ID"], how="left")
    .withColumn("GFR_STAGE", lab_gfr_label())
    .withColumn("UACR_STAGE", lab_uacr_label())
)
_matched = df_prep_b.filter(F.col("TEST_TYPE").isNotNull())
print(f"Rx claims: {_n_rx:,} | after join: {df_prep_b.count():,} (more = claims with several same-day tests) | "
      f"rows with a same-day test: {_matched.count():,}")
display(_matched.orderBy(NPI_COL, SVC_COL).limit(50))
display(_matched.groupBy("TEST_TYPE", "GFR_STAGE", "UACR_STAGE", "TREATMENT_CATEGORY").count()
        .orderBy("TEST_TYPE", "GFR_STAGE", "UACR_STAGE", "TREATMENT_CATEGORY"))

### 2.3 Family C: Rx claim + patient stage x treatments that month
Each Rx claim is joined to **every** row of the patient for that month in table 1.4, labelled with the patient's GFR / UACR stage. Because that table pairs every category with every brand, a claim of a patient on n treatments appears n x n times: each category column counts the claim n times (once per brand) and each brand column n times (once per category). This is how the original works and is kept as is in the refactor; worth raising with Wenjie.

Counted as: claims per stage x treatment category (e.g. `TOTAL_RX_CLAIMS_PATIENT_WITH_DX_GFR_STAGE_3_WITH_SGLT2`) and stage x brand for the same 8 brands. Rows with no treatment (`CATEGORY` null) are not counted.

Tables below: claim counts before/after the join (shows how much claims multiply), the claim with the most rows as an example, and rows per GFR stage x treatment category.

In [0]:
df_prep_c = (
    df_rx_m.select(NPI_COL, "PATIENT_ID", SVC_COL, "SVC_DT_M")
    .withColumn("_CLAIM_ROW", F.monotonically_increasing_id())
    .join(pat_uacr.select("PATIENT_ID", F.to_date("SVC_DT_M").alias("SVC_DT_M"),
                          "GFR_DX_STAGE", "UACR_STAGE", "CATEGORY", "BRAND"),
          on=["PATIENT_ID", "SVC_DT_M"], how="left")
    .withColumn("GFR_STAGE", gfr_label(F.col("GFR_DX_STAGE")))
    .withColumn("UACR_STAGE", uacr_label(F.col("UACR_STAGE")))
)
_counted_c = df_prep_c.filter(F.col("CATEGORY").isNotNull())
print(f"Rx claims: {_n_rx:,} | after join: {df_prep_c.count():,} | rows counted (CATEGORY not null): "
      f"{_counted_c.count():,}")

# Example: the Rx claim that turned into the most rows
_top = df_prep_c.groupBy("_CLAIM_ROW").count().orderBy(F.desc("count")).first()["_CLAIM_ROW"]
display(df_prep_c.filter(F.col("_CLAIM_ROW") == _top).drop("_CLAIM_ROW"))
display(_counted_c.groupBy("GFR_STAGE", "CATEGORY").count().orderBy("GFR_STAGE", F.desc("count")))

## 3. The original output (`final_claims_monthly`)
After counting, the original combines the families on NPI x month (missing = 0), fills months without claims, adds the last 1/3/6/12-month sums, `RUNNING_` totals and `PCT_` shares (each column / `TOTAL_RX_CLAIMS`), keeps months from 2022-01, maps NPI to `PROVIDER_ID` and writes one row per provider per month.

Tables below: row and column counts, columns grouped by family and type, and a few columns for the test HCPs' providers.

In [0]:
import os
import pandas as pd
# Wenjie's 3.0 output was missing on 2026-10-26 (volume cleaned). Show an empty frame instead of failing.
_orig_path = _NOTEBOOK_OUTPUT + "final_claims_monthly"
if os.path.exists(_orig_path):
    df_orig = spark.read.format("delta").load(_orig_path)
else:
    print(f"SKIPPED — 3.0 table missing, rerun when restored: {_orig_path}")
    df_orig = spark.createDataFrame([], "PROVIDER_ID string, SVC_DT_M timestamp")
_orig_cols = df_orig.columns

def _family(c):
    if "_PATIENT_WITH_TEST_" in c: return "B. same-day lab test"
    if "_PATIENT_WITH_DX_" in c:   return "C. stage x treatment"
    if "_PATIENT_WITH_" in c:      return "A. patient CKD stage"
    if "TOTAL_RX_CLAIMS" in c:     return "Total Rx claims"
    return "Keys"

def _kind(c):
    if c.startswith("PCT_"):     return "PCT_"
    if c.startswith("RUNNING_"): return "RUNNING_"
    if "_IN_LAST_" in c:         return "_IN_LAST_N_MONTH"
    return "base month"

_fam = pd.DataFrame({"column": _orig_cols})
_fam["family"] = _fam["column"].map(_family)
_fam["type"]   = _fam["column"].map(_kind)
print(f"final_claims_monthly: {df_orig.count():,} rows | {len(_orig_cols):,} columns")
display(df_orig.agg(F.min("SVC_DT_M").alias("first_month"), F.max("SVC_DT_M").alias("last_month"),
                    F.countDistinct("PROVIDER_ID").alias("providers")))
display(_fam.pivot_table(index="family", columns="type", values="column", aggfunc="count", fill_value=0).reset_index())

# Test HCPs' providers: keys, totals and one example column per family
_test = get_test_npis()
_map  = spark.read.format("delta").load(_QUERY_OUTPUT + HCP_NPI_MAPPING_TABLE)
_provs = [r[0] for r in (_map if _test is None else _map.filter(F.col(NPI_COL).isin(_test)))
          .select("PROVIDER_ID").distinct().collect()]
_show = ["PROVIDER_ID", "SVC_DT_M", "TOTAL_RX_CLAIMS", "TOTAL_RX_CLAIMS_IN_LAST_3_MONTH", "RUNNING_TOTAL_RX_CLAIMS",
         "TOTAL_RX_CLAIMS_PATIENT_WITH_GFR_STAGE_3", "TOTAL_RX_CLAIMS_PATIENT_WITH_GFR_STAGE_UNKNOWN",
         "TOTAL_RX_CLAIMS_PATIENT_WITH_TEST_GFR_STAGE_3_WITH_SGLT2",
         "TOTAL_RX_CLAIMS_PATIENT_WITH_DX_GFR_STAGE_3_WITH_SGLT2",
         "PCT_TOTAL_RX_CLAIMS_PATIENT_WITH_GFR_STAGE_3"]
_show = [c for c in _show if c in _orig_cols]
display(df_orig.filter(F.col("PROVIDER_ID").isin(_provs)).select(*_show).orderBy("PROVIDER_ID", "SVC_DT_M").limit(200))
# Full column list with family and type
display(_fam)

## 4. Cohort features (the refactor)

Same feature logic as the original, rebuilt per cohort (windows ending at each cohort cutoff, as in notebooks 01–03). To keep it fast with ~1,700 count columns, the work is split in two steps:

1. **Feature rows (once):** every Rx claim is turned into the list of feature names it adds 1 to, e.g. `TOTAL_RX_CLAIMS`, `..._PATIENT_WITH_GFR_STAGE_3`, `..._PATIENT_WITH_DX_GFR_STAGE_3_WITH_SGLT2`. The joins and labels are exactly those of sections 2.1–2.3. The result is a long table (NPI, service date, feature name, count).
2. **Per cohort (`build_cohort`, reference):** keep rows up to the cutoff, tag each row with the windows it falls in (last 1/3/6/12 months, and `RUNNING_` = all history), and pivot once into one row per NPI. Then `PCT_` = each count / `TOTAL_RX_CLAIMS` in the same window (0.0 when the total is 0).
3. **All cohorts (`build_all_cohorts_fast`, used for the output):** the feature rows are summed per NPI x cohort month x feature, each cohort month is tagged with every (cohort, window) it belongs to, and one pivot builds all cohorts (same approach as NB01-NB03, shared engine in `00_cohort_config`). The check cell compares it with `build_cohort` on the first and last cohort.

Rules kept from the original:
* Family B only counts claims that have a same-day lab test (the others have no month in the original). Their stage/drug combinations still become columns (all 0), as in the original pivot.
* Family C counts every (category, brand) row of the patient-month, including the n x n repetition.
* Column names use the original cleaning (no parenthesis removal). Base-month columns of the original are the `_IN_LAST_1_MONTH` columns here.
* The list of feature columns is taken from the data once, so every cohort has the same columns.

In [0]:
import re, time
import numpy as np
import pandas as pd

CLAIMS_LOOKBACK_WINDOWS = HCP_LOOKBACK_WINDOWS["claims"]   # original last_i_month_list (1, 3, 6, 12)
TOTAL_COL = "TOTAL_RX_CLAIMS"
BRANDS = HCP_BRANDS                               # original brand filter (00_cohort_config)
_PFX_A = "TOTAL_RX_CLAIMS_PATIENT_WITH_"          # family A
_PFX_B = _PFX_A + "TEST_"                          # family B
_PFX_C = _PFX_A + "DX_"                            # family C
# Original NB04 column cleaning, in order (no parenthesis removal, unlike clean_col_name)
_CLEAN_STEPS = [(", ", " "), ("+", "and"), ("/", " or"), ("-", "_"), (" ", "_"), ("__", "_")]

def clean_name_04(c):
    for a, b in _CLEAN_STEPS:
        c = F.regexp_replace(c, re.escape(a), b)
    return F.upper(c)

def _feat(prefix, sep, *cols):
    """prefix + cleaned concat_ws(sep, cols) -- same as the original group_combined + rename."""
    return F.concat(F.lit(prefix), clean_name_04(F.concat_ws(sep, *[F.col(x) for x in cols])))


def build_feature_rows(df_rx_in, pat_gfr_df, pat_uacr_df, lab_df, offset_days):
    """One row per (NPI, SVC_DT, FEATURE) with the number of times claims add to that feature.
    Joins and labels are the original 2.1 / 2.2 / 2.3 cells; the patient-table month is the
    claim's cohort month (= calendar month at offset 0)."""
    rx = (df_rx_in.select(NPI_COL, "PATIENT_ID", SVC_COL, "TREATMENT_CATEGORY", "CATEGORY")
          .withColumn("_M", assign_cohort_month(F.col(SVC_COL), offset_days)))

    # Total + family A: patient GFR / UACR stage that month (left join, one row per Rx claim)
    _g = pat_gfr_df.select("PATIENT_ID", F.to_date("SVC_DT_M").alias("_M"), "GFR_DX_STAGE",
                           F.col("UACR_STAGE").alias("_U"))
    a = (rx.join(_g, on=["PATIENT_ID", "_M"], how="left")
         .withColumn("GFR_STAGE", gfr_label(F.col("GFR_DX_STAGE")))
         .withColumn("UACR_STAGE", uacr_label(F.col("_U")))
         .select(NPI_COL, SVC_COL, F.lit(1).alias("_N"),
                 F.array(F.lit(TOTAL_COL),
                         _feat(_PFX_A, "", "GFR_STAGE"),
                         _feat(_PFX_A, "_AND_", "GFR_STAGE", "UACR_STAGE")).alias("_F")))

    # Family B: same-day lab test. Only matched claims count (_N = 1); unmatched claims only
    # contribute their column names (_N = 0), as in the original pivot.
    _l = lab_df.select("PATIENT_ID", SVC_COL, "TEST_TYPE", "STAGE").withColumn("_LAB", F.lit(1))
    b = (rx.join(_l, on=[SVC_COL, "PATIENT_ID"], how="left")
         .withColumn("GFR_STAGE", lab_gfr_label())
         .withColumn("UACR_STAGE", lab_uacr_label())
         .withColumn("_BR", F.coalesce(F.col("CATEGORY").isin(BRANDS), F.lit(False)))
         .select(NPI_COL, SVC_COL, F.coalesce(F.col("_LAB"), F.lit(0)).alias("_N"),
                 F.array(_feat(_PFX_B, "_WITH_", "GFR_STAGE", "TREATMENT_CATEGORY"),
                         F.when(F.col("_BR"), _feat(_PFX_B, "_WITH_", "GFR_STAGE", "CATEGORY")),
                         _feat(_PFX_B, "_WITH_", "UACR_STAGE", "TREATMENT_CATEGORY"),
                         F.when(F.col("_BR"), _feat(_PFX_B, "_WITH_", "UACR_STAGE", "CATEGORY"))).alias("_F")))

    # Family C: patient stage x every (category, brand) row of the patient-month
    _u = pat_uacr_df.select("PATIENT_ID", F.to_date("SVC_DT_M").alias("_M"), "GFR_DX_STAGE",
                            F.col("UACR_STAGE").alias("_U"), F.col("CATEGORY").alias("_CAT"),
                            F.col("BRAND").alias("_BRAND"))
    c = (rx.select(NPI_COL, "PATIENT_ID", SVC_COL, "_M")
         .join(_u, on=["PATIENT_ID", "_M"], how="inner")
         .withColumn("GFR_STAGE", gfr_label(F.col("GFR_DX_STAGE")))
         .withColumn("UACR_STAGE", uacr_label(F.col("_U")))
         .withColumn("_HC", F.col("_CAT").isNotNull())
         .withColumn("_HB", F.coalesce(F.col("_BRAND").isin(BRANDS), F.lit(False)))
         .select(NPI_COL, SVC_COL, F.lit(1).alias("_N"),
                 F.array(F.when(F.col("_HC"), _feat(_PFX_C, "_WITH_", "GFR_STAGE", "_CAT")),
                         F.when(F.col("_HB"), _feat(_PFX_C, "_WITH_", "GFR_STAGE", "_BRAND")),
                         F.when(F.col("_HC"), _feat(_PFX_C, "_WITH_", "UACR_STAGE", "_CAT")),
                         F.when(F.col("_HB"), _feat(_PFX_C, "_WITH_", "UACR_STAGE", "_BRAND"))).alias("_F")))

    return (a.unionByName(b).unionByName(c)
            .select(NPI_COL, SVC_COL, "_N", F.explode("_F").alias("FEATURE"))
            .filter(F.col("FEATURE").isNotNull())
            .groupBy(NPI_COL, SVC_COL, "FEATURE").agg(F.sum("_N").alias("N")))


# Staging prefix for the shared materialize() in 00_cohort_config (in memory in validation mode,
# _STAGE_PREFIX + name in production).
_STAGE_PREFIX = _NOTEBOOK_OUTPUT + "claims_monthly_feature_rows_"


# ---------- Cohorts to process: every cohort whose cutoff is within the data of all HCP inputs ----------
DATA_MAX_DT   = df_rx_raw.agg(F.max(SVC_COL)).first()[0]
COHORT_LIST   = get_hcp_cohort_months()   # common end date of all HCP inputs (00_cohort_config); DATA_MAX_DT kept for parity
COHORT_BOUNDS = {m: get_cohort_bounds(m) for m in COHORT_LIST}
COHORT_IDS    = [COHORT_BOUNDS[m]["cohort_id"] for m in COHORT_LIST]
PREVIEW_COHORT = COHORT_MONTH if COHORT_MONTH in COHORT_LIST else COHORT_LIST[0]
cb_preview     = COHORT_BOUNDS[PREVIEW_COHORT]
print(f"Rx data through {DATA_MAX_DT} -> {len(COHORT_LIST)} complete cohorts: "
      f"{COHORT_LIST[0]} .. {COHORT_LIST[-1]} | preview cohort: {PREVIEW_COHORT}")

# ---------- Feature rows for this run ----------
# Built once from the full Rx history, so the fixed feature list (columns) does not depend on
# DATA_START_DATE. Then the team rule (3.1, option B): every count, RUNNING_ included, uses claims
# from DATA_START_DATE on. Every feature row carries its Rx claim's SVC_DT, so filtering the rows
# is the same as filtering the claims. The offset-0 parity keeps the full history (as NB01-NB03).
_t0 = time.time()
df_feat_all = materialize(build_feature_rows(df_rx, pat_gfr, pat_uacr, df_lab, OFFSET_DAYS), "main")
df_feat = df_feat_all.filter(F.col(SVC_COL) >= F.lit(DATA_START_DATE))

# Fixed feature list: same columns for every cohort (from the full history)
FEATURE_VALUES = sorted(r[0] for r in df_feat_all.select("FEATURE").distinct().collect())
_fam_counts = pd.Series(["B" if v.startswith(_PFX_B) else "C" if v.startswith(_PFX_C)
                         else "A" if v.startswith(_PFX_A) else "Total" for v in FEATURE_VALUES]).value_counts()
print(f"Feature rows: {df_feat.count():,} (NPI x day x feature) in {time.time() - _t0:,.0f}s")
print(f"Feature columns per window: {len(FEATURE_VALUES)} -> " +
      ", ".join(f"{k}: {v}" for k, v in _fam_counts.sort_index().items()))

In [0]:
# ---- Display: the long feature-row table (what gets counted per NPI, day and feature) ----
_fam_col = (F.when(F.col("FEATURE").startswith(_PFX_B), "B. same-day lab test")
             .when(F.col("FEATURE").startswith(_PFX_C), "C. stage x treatment")
             .when(F.col("FEATURE").startswith(_PFX_A), "A. patient CKD stage")
             .otherwise("Total Rx claims"))
# 1) Rows and counts per family (N = claims added; family B rows with N = 0 only carry column names)
display(df_feat.withColumn("FAMILY", _fam_col).groupBy("FAMILY")
        .agg(F.count("*").alias("rows"), F.sum("N").alias("claims_counted"),
             F.countDistinct("FEATURE").alias("feature_columns")).orderBy("FAMILY"))
# 2) One NPI on one day: every feature its claims add to
_ex = df_feat.filter(F.col("FEATURE").startswith(_PFX_C)).orderBy(F.desc("N")).first()
display(df_feat.filter((F.col(NPI_COL) == _ex[NPI_COL]) & (F.col(SVC_COL) == _ex[SVC_COL]))
        .withColumn("FAMILY", _fam_col).orderBy("FAMILY", "FEATURE"))

In [0]:
# ---- One cohort: windows + RUNNING_ in a single pivot, then PCT_ and metadata ----
def build_cohort(cb, df_feat_in=None, feature_values=None):
    """All features for one cohort, one row per NPI with any Rx claim up to the cutoff.
    df_feat_in / feature_values: feature rows and column list (default: this run's)."""
    L    = df_feat if df_feat_in is None else df_feat_in
    vals = FEATURE_VALUES if feature_values is None else feature_values
    wins = CLAIMS_LOOKBACK_WINDOWS
    cutoff = F.to_date(F.lit(cb["cutoff"].strftime("%Y-%m-%d")))

    # Each row goes to RUNNING_ and to every window whose (start, cutoff] contains its date
    _suffixes = F.array(F.lit("RUNNING"), *[
        F.when(F.col(SVC_COL) > F.to_date(F.lit(cb["window_bounds"][w]["start"])), F.lit(f"_IN_LAST_{w}_MONTH"))
        for w in wins])
    long = (L.filter(F.col(SVC_COL) <= cutoff)
            .select(NPI_COL, "FEATURE", "N", F.explode(_suffixes).alias("_S"))
            .filter(F.col("_S").isNotNull())
            .withColumn("_COL", F.when(F.col("_S") == "RUNNING", F.concat(F.lit("RUNNING_"), F.col("FEATURE")))
                                 .otherwise(F.concat(F.col("FEATURE"), F.col("_S"))))
            .groupBy(NPI_COL, "_COL").agg(F.sum("N").alias("N")))

    window_cols  = [f"{v}_IN_LAST_{w}_MONTH" for w in wins for v in vals]
    running_cols = [f"RUNNING_{v}" for v in vals]
    wide = (long.groupBy(NPI_COL).pivot("_COL", window_cols + running_cols)
            .agg(F.first("N")).na.fill(0))

    # PCT_: feature / TOTAL_RX_CLAIMS in the same window (0.0 when the total is 0); none for RUNNING_
    pct = []
    for w in wins:
        den = F.col(f"{TOTAL_COL}_IN_LAST_{w}_MONTH")
        pct += [F.when(den != 0, F.col(f"{v}_IN_LAST_{w}_MONTH") / den).otherwise(F.lit(0.0))
                 .alias(f"PCT_{v}_IN_LAST_{w}_MONTH") for v in vals if v != TOTAL_COL]

    return (wide.select("*", *pct)
            .withColumn("COHORT_ID", F.lit(cb["cohort_id"]))
            .withColumn("COHORT_MONTH", F.lit(cb["cohort_month"]))
            .withColumn("OFFSET_DAYS", F.lit(cb["offset_days"]))
            .withColumn("CUTOFF_DATE", F.lit(cb["cutoff"].strftime("%Y-%m-%d")))
            .withColumn("SVC_DT_M", F.to_timestamp(F.lit(f"{cb['cohort_month']}-01")))
            .withColumn("PULL_DT", F.current_date()))


# ---- Preview on one cohort ----
_t0 = time.time()
claims_preview = build_cohort(cb_preview)
_pv_cols = claims_preview.columns
print(f"Preview {cb_preview['cohort_id']} (cutoff {cb_preview['cutoff']:%Y-%m-%d}) | "
      f"NPIs: {claims_preview.count():,} | columns: {len(_pv_cols):,} "
      f"(windows: {builtins.sum('_IN_LAST_' in c and not c.startswith('PCT_') for c in _pv_cols)}, "
      f"RUNNING_: {builtins.sum(c.startswith('RUNNING_') for c in _pv_cols)}, "
      f"PCT_: {builtins.sum(c.startswith('PCT_') for c in _pv_cols)}) | {time.time() - _t0:,.0f}s")

In [0]:
# ---- Display: preview cohort, keys + totals + one example column per family ----
_ex_cols = [c for c in [
    f"{TOTAL_COL}_IN_LAST_1_MONTH", f"{TOTAL_COL}_IN_LAST_3_MONTH", f"{TOTAL_COL}_IN_LAST_12_MONTH", f"RUNNING_{TOTAL_COL}",
    "TOTAL_RX_CLAIMS_PATIENT_WITH_GFR_STAGE_UNKNOWN_IN_LAST_12_MONTH",
    "TOTAL_RX_CLAIMS_PATIENT_WITH_GFR_STAGE_4_IN_LAST_12_MONTH",
    "PCT_TOTAL_RX_CLAIMS_PATIENT_WITH_GFR_STAGE_4_IN_LAST_12_MONTH",
    "TOTAL_RX_CLAIMS_PATIENT_WITH_TEST_GFR_STAGE_2_WITH_MRA_IN_LAST_12_MONTH",
    "TOTAL_RX_CLAIMS_PATIENT_WITH_DX_GFR_STAGE_4_WITH_SGLT2_IN_LAST_12_MONTH",
    "RUNNING_TOTAL_RX_CLAIMS_PATIENT_WITH_DX_GFR_STAGE_4_WITH_SGLT2"] if c in _pv_cols]
display(claims_preview.select(NPI_COL, "COHORT_ID", "CUTOFF_DATE", *_ex_cols).orderBy(NPI_COL))

In [0]:
# ---- Fast engine (pattern A, as NB01-NB03): all cohorts in ONE query ----
# Same columns, order, types and values as build_cohort (checked in the next cell):
# 1) feature rows -> (NPI, cohort month, FEATURE, N), once per feature-row table (materialized)
# 2) one pivot by NPI x cohort through the window tags of 00_cohort_config, then PCT_ and metadata.
OUTPUT_COLS = build_cohort(cb_preview).columns   # schema only, nothing is computed
_FEAT_MONTHS = {}

def get_feat_months(df_feat_in, offset_days, name):
    if name not in _FEAT_MONTHS:
        _t = time.time()
        _FEAT_MONTHS[name] = materialize(feature_rows_to_months(df_feat_in, NPI_COL, offset_days), f"months_{name}")
        print(f"Feature rows by cohort month ({name}) built in {time.time() - _t:,.0f}s")
    return _FEAT_MONTHS[name]

def build_all_cohorts_fast(cbs, df_feat_in=None, feature_values=None, name="run"):
    """build_cohort for every cohort in cbs, in one query (one row per NPI x COHORT_ID).
    df_feat_in / feature_values / name: another feature-row table (e.g. the offset-0 parity rows)."""
    L    = df_feat if df_feat_in is None else df_feat_in
    vals = FEATURE_VALUES if feature_values is None else feature_values
    wins = CLAIMS_LOOKBACK_WINDOWS
    off  = cbs[0]["offset_days"]
    wide = build_all_cohorts_from_months(get_feat_months(L, off, f"{name}_offset{off}"), cbs, NPI_COL, vals, wins)
    pct = []
    for w in wins:
        den = F.col(f"{TOTAL_COL}_IN_LAST_{w}_MONTH")
        pct += [F.when(den != 0, F.col(f"{v}_IN_LAST_{w}_MONTH") / den).otherwise(F.lit(0.0))
                 .alias(f"PCT_{v}_IN_LAST_{w}_MONTH") for v in vals if v != TOTAL_COL]
    out = add_cohort_meta(wide.select("*", *pct), cbs)
    cols = OUTPUT_COLS if vals is FEATURE_VALUES else build_cohort(cbs[0], L, vals).columns
    return out.select(*cols)

print(f"Fast engine ready: {len(OUTPUT_COLS):,} output columns per cohort, {len(COHORT_LIST)} cohorts in one query")

In [0]:
# ---- Check: the all-cohorts query gives exactly what build_cohort gives (first and last cohort) ----
# Nothing is written.
_check = [COHORT_LIST[0], COHORT_LIST[-1]]
_t0 = time.time()
_pd_fast = build_all_cohorts_fast([COHORT_BOUNDS[m] for m in _check]).toPandas()
print(f"All-cohorts query ({len(_check)} cohorts): {time.time() - _t0:,.0f}s")

_rows = []
for m in _check:
    cb = COHORT_BOUNDS[m]
    _t0 = time.time()
    _one  = build_cohort(cb)
    pd_one = _one.toPandas().set_index(NPI_COL)
    _t_one = time.time() - _t0
    pd_all = _pd_fast[_pd_fast["COHORT_ID"] == cb["cohort_id"]].set_index(NPI_COL)

    same_order = list(pd_one.columns) == list(pd_all.columns)
    same_types = _one.dtypes == build_all_cohorts_fast([cb]).dtypes
    cols = [c for c in pd_one.columns if c in pd_all.columns and c != "PULL_DT"]
    num  = [c for c in cols if pd.api.types.is_numeric_dtype(pd_one[c])]
    txt  = [c for c in cols if c not in num]
    common = sorted(set(pd_one.index) & set(pd_all.index))
    bad = ~np.isclose(pd_one.loc[common, num].astype(float).values, pd_all.loc[common, num].astype(float).values,
                      rtol=1e-9, atol=1e-9, equal_nan=True)
    bad_txt = int((pd_one.loc[common, txt].astype(str).values != pd_all.loc[common, txt].astype(str).values).sum())
    _rows.append({"cohort": cb["cohort_id"], "npis_per_cohort": len(pd_one), "npis_fast": len(pd_all),
                  "npis_common": len(common), "same_column_order": same_order, "same_types": same_types,
                  "cells_compared": int(bad.size) + len(common) * len(txt),
                  "mismatching_cells": int(bad.sum()) + bad_txt, "per_cohort_seconds": builtins.round(_t_one)})

df_fast_check = pd.DataFrame(_rows)
display(df_fast_check)
_ok = (df_fast_check["mismatching_cells"].sum() == 0 and df_fast_check["same_column_order"].all()
       and df_fast_check["same_types"].all()
       and (df_fast_check["npis_per_cohort"] == df_fast_check["npis_common"]).all()
       and (df_fast_check["npis_fast"] == df_fast_check["npis_common"]).all())
print("PASS: fast engine = build_cohort" if _ok else "CHECK: fast engine differs from build_cohort")

## 5. Build and Write All Cohorts
Builds every cohort in `COHORT_LIST` with the fast engine (one query) and writes them in **one write** (`replaceWhere COHORT_ID IN (...)`, other cohorts and offsets are kept) to `claims_monthly_cohort` (NPI level), then the provider-level copy `final_claims_monthly_cohort`. Wenjie's `final_claims_monthly` is not touched.

In [0]:
OUTPUT_PATH = _NOTEBOOK_OUTPUT + "claims_monthly_cohort"

# All cohorts in one query and ONE write (fast engine, as NB01-NB03). replaceWhere keeps every
# other cohort / offset already in the table.
_ids_sql = ", ".join(f"'{c}'" for c in COHORT_IDS)
_t_all = time.time()
build_all_cohorts_fast([COHORT_BOUNDS[m] for m in COHORT_LIST]).write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .option("replaceWhere", f"COHORT_ID IN ({_ids_sql})") \
    .partitionBy("COHORT_ID") \
    .save(OUTPUT_PATH)

print(f"{len(COHORT_LIST)} cohorts ({COHORT_IDS[0]} .. {COHORT_IDS[-1]}) written to {OUTPUT_PATH} "
      f"in one write, {(time.time() - _t_all) / 60:,.1f} min")

In [0]:
# Read the written table directly (independent of the preview variables)
df_all_cohorts = spark.read.format("delta").load(OUTPUT_PATH)

_ids_in_table  = sorted(r[0] for r in df_all_cohorts.select("COHORT_ID").distinct().collect())
_missing       = sorted(set(COHORT_IDS) - set(_ids_in_table))
_extra         = sorted(set(_ids_in_table) - set(COHORT_IDS))
df_all_cohorts = df_all_cohorts.filter(F.col("COHORT_ID").isin(COHORT_IDS))
_n_rows = df_all_cohorts.count()
_n_keys = df_all_cohorts.select(NPI_COL, "COHORT_ID").distinct().count()
_n_npis = df_all_cohorts.select(NPI_COL).distinct().count()
_nulls  = df_all_cohorts.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_all_cohorts.columns]).first()
_null_cells = builtins.sum(v or 0 for v in _nulls)

print(f"Cohorts expected: {len(COHORT_IDS)} | in table: {len(_ids_in_table)} | "
      f"missing: {_missing or 'none'} | extra (not in this run): {_extra or 'none'}")
print(f"Rows: {_n_rows:,} | distinct (NPI, COHORT_ID): {_n_keys:,} | duplicates: {_n_rows - _n_keys} | "
      f"NPIs: {_n_npis} | Columns: {len(df_all_cohorts.columns):,} | null cells: {_null_cells:,}")
display(df_all_cohorts.groupBy("COHORT_ID", "CUTOFF_DATE").count().orderBy("COHORT_ID"))

# Keys + totals per NPI x cohort (the full table has ~3,000 columns; open OUTPUT_PATH for all of them)
_key_cols = [NPI_COL, "COHORT_ID", "COHORT_MONTH", "OFFSET_DAYS", "CUTOFF_DATE", "SVC_DT_M", "PULL_DT"]
_tot_cols = [f"{TOTAL_COL}_IN_LAST_{w}_MONTH" for w in CLAIMS_LOOKBACK_WINDOWS] + [f"RUNNING_{TOTAL_COL}"]
display(df_all_cohorts.select(*_key_cols, *_tot_cols).orderBy(NPI_COL, "COHORT_ID"))

In [0]:
# Same mapping as the original (NPI -> PROVIDER_ID, NPI dropped). Wenjie's final_claims_monthly is NOT touched.
hcp_npi_mapping = spark.read.format("delta").load(_QUERY_OUTPUT + HCP_NPI_MAPPING_TABLE)
hcp_npi_mapping_df = hcp_npi_mapping.select("NPI_NUMBER", "PROVIDER_ID").distinct()

PROVIDER_OUTPUT_PATH = _NOTEBOOK_OUTPUT + "final_claims_monthly_cohort"
(spark.read.format("delta").load(OUTPUT_PATH)
    .filter(F.col("COHORT_ID").isin(COHORT_IDS))
    .join(hcp_npi_mapping_df, on="NPI_NUMBER", how="left")
    .drop("NPI_NUMBER")
    .write.format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .option("replaceWhere", "COHORT_ID IN (" + ", ".join(f"'{c}'" for c in COHORT_IDS) + ")")
    .partitionBy("COHORT_ID")
    .save(PROVIDER_OUTPUT_PATH))

_prov = spark.read.format("delta").load(PROVIDER_OUTPUT_PATH).filter(F.col("COHORT_ID").isin(COHORT_IDS))
print(f"Written to {PROVIDER_OUTPUT_PATH} ({len(COHORT_IDS)} cohorts) | rows: {_prov.count():,} | "
      f"null PROVIDER_ID: {_prov.filter(F.col('PROVIDER_ID').isNull()).count():,} | "
      f"duplicate (PROVIDER_ID, COHORT_ID): {_prov.count() - _prov.select('PROVIDER_ID', 'COHORT_ID').distinct().count():,}")
display(_prov.select("PROVIDER_ID", "COHORT_ID", "CUTOFF_DATE", *_tot_cols).orderBy("PROVIDER_ID", "COHORT_ID"))

## 6. Validation

1. **Offset = 0 comparison with the original (every cohort):** rebuild the features at offset 0 for the test HCPs with the same inputs as the original (the 3.0 patient tables, calendar months), map NPI to `PROVIDER_ID` and compare every column with Wenjie's `final_claims_monthly` for each month. Original base-month columns are compared with our `_IN_LAST_1_MONTH` columns; original nulls count as 0. Original columns we do not have (stage/drug combinations no test HCP ever had) must be 0 for the test providers.
2. **HCP trace:** for the busiest test HCP, rebuild every feature in pandas from the raw tables (separate code: pandas joins and Python string cleaning) and compare every window and `RUNNING_` count with the written table, for every cohort.

In [0]:
# ---- Validation 1: offset = 0 parity vs original final_claims_monthly, every cohort ----
import os
_npis = get_test_npis()
# The parity needs Wenjie's 3.0 output + 3.0 patient tables. All were missing on 2026-10-26
# (volume cleaned); skip instead of failing so Run all still completes.
_ref_missing = [p for p in [_NOTEBOOK_OUTPUT + "final_claims_monthly",
                            _QUERY_OUTPUT + get_patient_table_30("gfr_dx_stage"),
                            _QUERY_OUTPUT + get_patient_table_30("uacr_stage")]
                if not os.path.exists(p)]
if _npis is None:
    print("Production mode: parity check skipped (runs on the test NPIs only)")
elif _ref_missing:
    print("SKIPPED — 3.0 reference table(s) missing, rerun when restored:\n  " + "\n  ".join(_ref_missing))
else:
    _t_par = time.time()
    # Same inputs as the original: 3.0 patient tables (calendar months), only the test HCPs' patients
    _pid_df = spark.createDataFrame(pd.DataFrame({"PATIENT_ID": [r[0] for r in _rx_patients.collect()]}))
    def _patients_only(table):
        return (spark.read.format("delta").load(_QUERY_OUTPUT + table)
                .join(F.broadcast(_pid_df), on="PATIENT_ID", how="left_semi"))
    _gfr30  = _patients_only(get_patient_table_30("gfr_dx_stage")).select(
        "PATIENT_ID", "SVC_DT_M", "GFR_DX_STAGE", "UACR_STAGE")
    _uacr30 = _patients_only(get_patient_table_30("uacr_stage")).select(
        "PATIENT_ID", "SVC_DT_M", "GFR_DX_STAGE", "UACR_STAGE", "CATEGORY", "BRAND")
    df_feat0 = materialize(build_feature_rows(df_rx, _gfr30, _uacr30, df_lab, 0), "offset0")
    FEATURE_VALUES0 = sorted(r[0] for r in df_feat0.select("FEATURE").distinct().collect())
    print(f"Offset-0 feature rows: {df_feat0.count():,} | feature columns: {len(FEATURE_VALUES0)} "
          f"| {time.time() - _t_par:,.0f}s")

    _map_pd = hcp_npi_mapping_df.filter(F.col(NPI_COL).isin(_npis)).toPandas()
    _npi_to_prov = dict(zip(_map_pd[NPI_COL], _map_pd["PROVIDER_ID"]))
    _orig_all = (spark.read.format("delta").load(_NOTEBOOK_OUTPUT + "final_claims_monthly")
                 .filter(F.col("PROVIDER_ID").isin(list(_npi_to_prov.values())))
                 .withColumn("_M", F.date_format("SVC_DT_M", "yyyy-MM")))
    _orig_months  = {r[0] for r in _orig_all.select("_M").distinct().collect()}
    PARITY_MONTHS = [m for m in get_cohort_months(max_data_date=DATA_MAX_DT, offset_days=0) if m in _orig_months]
    pd_orig_all = _orig_all.filter(F.col("_M").isin(PARITY_MONTHS)).toPandas()
    print(f"Original rows for the test providers: {len(pd_orig_all):,} | months compared: {len(PARITY_MONTHS)}")

    _skip = {"SVC_DT_M", "PULL_DT", "_M", "PROVIDER_ID"}
    _orig_cols = [c for c in pd_orig_all.columns if c not in _skip]
    # Original column -> cohort-based column (base-month columns map to _IN_LAST_1_MONTH)
    _map = {oc: (oc if ("_IN_LAST_" in oc or oc.startswith("RUNNING_")) else f"{oc}_IN_LAST_1_MONTH")
            for oc in _orig_cols}
    _meta = {"COHORT_ID", "COHORT_MONTH", "OFFSET_DAYS", "CUTOFF_DATE", "SVC_DT_M", "PULL_DT", NPI_COL, "PROVIDER_ID"}

    def _num(df):
        return df.apply(pd.to_numeric, errors="coerce").fillna(0).astype(float)

    # All parity months in ONE query (same engine as the written output), compared month by month
    _cbs0 = {m: get_cohort_bounds(m, offset_days=0) for m in PARITY_MONTHS}
    pd_new_all = build_all_cohorts_fast(list(_cbs0.values()), df_feat0, FEATURE_VALUES0, name="parity").toPandas()

    summary, mismatches = [], []
    for m in PARITY_MONTHS:
        _t0 = time.time()
        cb0 = _cbs0[m]
        pd_new = pd_new_all[pd_new_all["COHORT_ID"] == cb0["cohort_id"]].copy()
        pd_new["PROVIDER_ID"] = pd_new[NPI_COL].map(_npi_to_prov)
        pd_new  = pd_new.set_index("PROVIDER_ID")
        pd_orig = pd_orig_all[pd_orig_all["_M"] == m].set_index("PROVIDER_ID")

        common    = sorted(set(pd_new.index) & set(pd_orig.index))
        only_orig = sorted(set(pd_orig.index) - set(pd_new.index))
        only_new  = sorted(set(pd_new.index) - set(pd_orig.index))
        present   = [oc for oc in _orig_cols if _map[oc] in pd_new.columns]
        absent    = [oc for oc in _orig_cols if _map[oc] not in pd_new.columns]
        extra_new = [c for c in pd_new.columns if c not in _meta and c not in set(_map.values())]

        o = _num(pd_orig.loc[common, present])
        n = _num(pd_new.loc[common, [_map[oc] for oc in present]])
        bad = ~np.isclose(o.values, n.values, rtol=1e-9, atol=1e-9)
        for i, j in list(zip(*np.where(bad)))[:50]:
            mismatches.append({"cohort": cb0["cohort_id"], "PROVIDER_ID": common[i], "original_col": present[j],
                               "new_col": _map[present[j]], "original": o.values[i, j], "new": n.values[i, j]})
        # Columns we do not build (combinations the test HCPs never had) must be 0 in the original
        absent_nonzero = int((_num(pd_orig.loc[common, absent]) != 0).values.sum()) if absent else 0
        # Providers only in the original (no Rx history yet that month) must be all zeros
        only_orig_nonzero = (int((_num(pd_orig.loc[only_orig, _orig_cols]) != 0).any(axis=1).sum())
                             if only_orig else 0)

        summary.append({"cohort_offset0": cb0["cohort_id"], "cutoff": f"{cb0['cutoff']:%Y-%m-%d}",
                        "providers_common": len(common), "providers_only_original": len(only_orig),
                        "only_original_nonzero": only_orig_nonzero, "providers_only_new": len(only_new),
                        "columns_compared": len(present), "original_cols_not_built": len(absent),
                        "not_built_nonzero_cells": absent_nonzero, "new_cols_not_in_original": len(extra_new),
                        "mismatching_cells": int(bad.sum()), "seconds": builtins.round(time.time() - _t0)})
        print(f"{cb0['cohort_id']}: {int(bad.sum())} mismatching cells, {len(present):,} columns compared "
              f"({time.time() - _t0:,.0f}s)")

    df_parity = pd.DataFrame(summary)
    display(df_parity)
    _ok = (df_parity[["mismatching_cells", "only_original_nonzero", "providers_only_new",
                      "not_built_nonzero_cells", "new_cols_not_in_original"]].values.sum() == 0)
    print(f"Cohorts compared: {len(df_parity)} | mismatching cells: {df_parity['mismatching_cells'].sum()} | "
          f"{'PASS' if _ok else 'CHECK MISMATCHES'} | total {(time.time() - _t_par) / 60:,.1f} min")
    if mismatches:
        _mm = pd.DataFrame(mismatches)
        _mm["family"] = np.select([_mm["original_col"].str.contains("_WITH_TEST_"),
                                   _mm["original_col"].str.contains("_WITH_DX_"),
                                   _mm["original_col"].str.contains("_PATIENT_WITH_")],
                                  ["B", "C", "A"], default="Total")
        display(_mm.groupby("family").size().reset_index(name="cells (first 50 per cohort)"))
        display(_mm)

In [0]:
# ---- Validation 2: HCP trace. Every feature rebuilt in pandas from the raw tables vs written table ----
TRACE_NPI = HCP_TRACE_NPIS["claims"]   # fixed trace HCP (00_cohort_config)
_off = OFFSET_DAYS

def _pm(s):
    """Cohort month in pandas (same rule as assign_cohort_month: the offset-th day starts the month)."""
    return ((s - pd.Timedelta(days=_off - 1)) if _off > 0 else s).dt.to_period("M").dt.to_timestamp()

def _clean_py(s):
    for a, b in _CLEAN_STEPS:
        s = s.replace(a, b)
    return s.upper()

def _ws(sep, *parts):
    return sep.join(p for p in parts if p is not None and not pd.isna(p))

_GFR_MAP  = {1: "GFR_STAGE_1", 2: "GFR_STAGE_2", 3: "GFR_STAGE_3", 4: "GFR_STAGE_4", 5: "GFR_STAGE_5"}
_UACR_MAP = {1: "UACR_STAGE_1", 2: "UACR_STAGE_2", 3: "UACR_STAGE_3"}
def _g(x): return _GFR_MAP.get(int(x), "GFR_STAGE_UNKNOWN") if pd.notna(x) else "GFR_STAGE_UNKNOWN"
def _u(x): return _UACR_MAP.get(int(x), "UACR_STAGE_UNKNOWN") if pd.notna(x) else "UACR_STAGE_UNKNOWN"

# Counted claims only: from DATA_START_DATE on, as the features (RUNNING_ included)
_rx_t = (df_rx.filter((F.col(NPI_COL) == TRACE_NPI) & (F.col(SVC_COL) >= F.lit(DATA_START_DATE)))
         .select("PATIENT_ID", SVC_COL, "TREATMENT_CATEGORY", "CATEGORY").toPandas())
_rx_t[SVC_COL] = pd.to_datetime(_rx_t[SVC_COL])
_rx_t["_M"] = _pm(_rx_t[SVC_COL])
_pl = _rx_t["PATIENT_ID"].unique().tolist()
_gfr_t = (pat_gfr.filter(F.col("PATIENT_ID").isin(_pl))
          .select("PATIENT_ID", F.to_date("SVC_DT_M").alias("_M"), F.col("GFR_DX_STAGE").cast("int").alias("G"),
                  F.col("UACR_STAGE").cast("int").alias("U")).toPandas())
_lab_t = df_lab.filter(F.col("PATIENT_ID").isin(_pl)).select("PATIENT_ID", SVC_COL, "TEST_TYPE", "STAGE").toPandas()
_uac_t = (pat_uacr.filter(F.col("PATIENT_ID").isin(_pl))
          .select("PATIENT_ID", F.to_date("SVC_DT_M").alias("_M"), F.col("GFR_DX_STAGE").cast("int").alias("G"),
                  F.col("UACR_STAGE").cast("int").alias("U"), F.col("CATEGORY").alias("CAT"), "BRAND").toPandas())
for _d in (_gfr_t, _uac_t):
    _d["_M"] = pd.to_datetime(_d["_M"])
_lab_t[SVC_COL] = pd.to_datetime(_lab_t[SVC_COL])

_rows = []   # (SVC_DT, FEATURE): one entry per +1
for r in _rx_t.merge(_gfr_t, on=["PATIENT_ID", "_M"], how="left").itertuples():
    gs, us = _g(r.G), _u(r.U)
    _rows += [(r.SVC_DT, TOTAL_COL), (r.SVC_DT, _PFX_A + _clean_py(gs)), (r.SVC_DT, _PFX_A + _clean_py(gs + "_AND_" + us))]
for r in _rx_t.merge(_lab_t, on=["PATIENT_ID", SVC_COL], how="inner").itertuples():
    gs = (_GFR_LAB_STAGES.get(r.STAGE) if r.TEST_TYPE == "GFR" else None) or "GFR_STAGE_UNSPECIFIED"
    us = (_UACR_LAB_STAGES.get(r.STAGE) if r.TEST_TYPE == "UACR" else None) or "UACR_STAGE_UNSPECIFIED"
    for s in (gs, us):
        _rows.append((r.SVC_DT, _PFX_B + _clean_py(_ws("_WITH_", s, r.TREATMENT_CATEGORY))))
        if r.CATEGORY in BRANDS:
            _rows.append((r.SVC_DT, _PFX_B + _clean_py(_ws("_WITH_", s, r.CATEGORY))))
for r in _rx_t[["PATIENT_ID", SVC_COL, "_M"]].merge(_uac_t, on=["PATIENT_ID", "_M"], how="inner").itertuples():
    for s in (_g(r.G), _u(r.U)):
        if pd.notna(r.CAT):
            _rows.append((r.SVC_DT, _PFX_C + _clean_py(_ws("_WITH_", s, r.CAT))))
        if r.BRAND in BRANDS:
            _rows.append((r.SVC_DT, _PFX_C + _clean_py(_ws("_WITH_", s, r.BRAND))))
_raw_feat = pd.DataFrame(_rows, columns=[SVC_COL, "FEATURE"])

_out = (spark.read.format("delta").load(OUTPUT_PATH)
        .filter((F.col(NPI_COL) == TRACE_NPI) & F.col("COHORT_ID").isin(COHORT_IDS))
        .toPandas().set_index("COHORT_ID"))

rows, bad = [], []
for m in COHORT_LIST:
    cb, cid = COHORT_BOUNDS[m], COHORT_BOUNDS[m]["cohort_id"]
    feat = _out.loc[cid] if cid in _out.index else pd.Series(dtype=float)
    upto = _raw_feat[_raw_feat[SVC_COL] <= pd.Timestamp(cb["cutoff"])]
    expected = {f"RUNNING_{k}": v for k, v in upto.groupby("FEATURE").size().items()}
    row = {"COHORT_ID": cid, "CUTOFF_DATE": f"{cb['cutoff']:%Y-%m-%d}"}
    for w in CLAIMS_LOOKBACK_WINDOWS:
        sel = upto[upto[SVC_COL] > pd.Timestamp(cb["window_bounds"][w]["start"])]
        expected.update({f"{k}_IN_LAST_{w}_MONTH": v for k, v in sel.groupby("FEATURE").size().items()})
        row[f"{w}M_raw"] = int((sel["FEATURE"] == TOTAL_COL).sum())
        row[f"{w}M_feature"] = int(feat.get(f"{TOTAL_COL}_IN_LAST_{w}_MONTH", 0) or 0)
    # Every written count column is checked (expected 0 when the raw rebuild has no claims for it)
    _count_cols = [c for c in _out.columns if ("_IN_LAST_" in c or c.startswith("RUNNING_")) and not c.startswith("PCT_")]
    _not_written = [k for k in expected if k not in _out.columns]
    n_bad = len(_not_written)
    for k in _not_written:
        bad.append({"COHORT_ID": cid, "feature": k, "raw": expected[k], "feature_value": "column missing"})
    for c in _count_cols:
        fv = int(feat.get(c, 0) or 0)
        if fv != int(expected.get(c, 0)):
            n_bad += 1
            bad.append({"COHORT_ID": cid, "feature": c, "raw": int(expected.get(c, 0)), "feature_value": fv})
    row.update({"RUNNING_raw": int((upto["FEATURE"] == TOTAL_COL).sum()),
                "RUNNING_feature": int(feat.get(f"RUNNING_{TOTAL_COL}", 0) or 0),
                "checks": len(_count_cols), "mismatches": n_bad})
    rows.append(row)

trace = pd.DataFrame(rows)
print(f"Trace NPI: {TRACE_NPI} | Rx claims: {len(_rx_t):,} | raw feature +1s: {len(_raw_feat):,} | "
      f"cohorts: {len(trace)} | checks: {trace['checks'].sum():,} | mismatches: {trace['mismatches'].sum()}")
display(trace)
if bad:
    display(pd.DataFrame(bad).head(50))